In [ ]:
# ----------------------------- Setup ----------------------------- #
%matplotlib tk
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import Button
from matplotlib.patches import Circle   # ← for δ visualization
from sage.all import factorial
import re

prec = 1000
RHP = RealField(prec)
CHP = ComplexField(prec)

# ----------------------------- Printing precision ----------------------------- #
print_digits = 20   # ← number of decimal digits shown for roots

# ----------------------------- Input ----------------------------- #
def get_polynomial_coeffs():
    """
    Read polynomial coefficients from user:
    - Accepts integers, decimals, scientific notation
    - Strips leading zeros
    - Rejects all-zero polynomials
    - RETURNS BOTH: high-precision values + original string representations
      (so the printed polynomial uses EXACTLY the digits the user typed)
    """
    pattern = re.compile(r'^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$')
    while True:
        s = input("Enter polynomial coefficients (highest degree first): ").strip()
        if not s:
            print("Error: No input provided.")
            continue
        tokens = s.split()
        coeffs_hp = []
        coeff_strings = []   # keep exact user input strings
        for t in tokens:
            t_clean = t.lstrip("0") or "0"
            if t_clean.startswith("."):
                t_clean = "0" + t_clean
            if not pattern.fullmatch(t_clean):
                print(f"Invalid number: '{t}'")
                break
            try:
                coeffs_hp.append(RHP(t_clean))
                coeff_strings.append(t)   # store EXACTLY what the user typed
            except Exception:
                print(f"Could not convert '{t}' to high-precision number.")
                break
        else:
            # Reject all-zero polynomial
            if all(c == 0 for c in coeffs_hp):
                print("Polynomial cannot be identically zero.")
                continue
            # Strip leading zeros to get true degree
            while coeffs_hp and coeffs_hp[0] == 0:
                coeffs_hp.pop(0)
                coeff_strings.pop(0)
            return coeffs_hp, coeff_strings   # ← returns TWO lists

coeffs_hp, coeff_strings = get_polynomial_coeffs()

# ----------------------------- Polynomial Construction ----------------------------- #
def build_polynomial_from_coeffs(coeffs_hp, RHP, CHP):
    """
    Safely construct polynomial from coefficients (highest degree first),
    avoiding exponent/order bugs.
    """
    R.<x> = RHP[]
    # Use Horner's method (robust, order-safe)
    f = R(0)
    for c in coeffs_hp:
        f = f * x + c
    fC = f.change_ring(CHP)
    return f, fC

# build polynomial
f, fC = build_polynomial_from_coeffs(coeffs_hp, RHP, CHP)

# ----------------------------- Clean Printing (using user strings) ----------------------------- #
def polynomial_to_string(coeffs_hp, coeff_strings, var="x"):
    if not coeff_strings:
        return "0"
    terms = []
    n = len(coeff_strings)
    for i, (c_str, c_hp) in enumerate(zip(coeff_strings, coeffs_hp)):
        power = n - i - 1
        # skip numerical zeros
        if abs(c_hp) < RHP('1e-30'):
            continue
        sign = "-" if c_str.startswith("-") else "+"
        fc_abs = c_str.lstrip("+-")
        # --- FIX: handle coefficient properly ---
        if power == 0:
            term = fc_abs
        elif power == 1:
            if fc_abs == "1":
                term = f"{var}"
            else:
                term = f"{fc_abs}*{var}"
        else:
            if fc_abs == "1":
                term = f"{var}^{power}"
            else:
                term = f"{fc_abs}*{var}^{power}"
        terms.append((sign, term))
    if not terms:
        return "0"
    # first term
    first_sign, first_term = terms[0]
    result = first_term if first_sign == "+" else f"-{first_term}"
    for sign, term in terms[1:]:
        result += f" {sign} {term}"
    return result

poly_str = polynomial_to_string(coeffs_hp, coeff_strings)
print(f"\nPolynomial: f(x) = {poly_str} = 0")

# ----------------------------- Roots ----------------------------- #
roots = fC.roots(multiplicities=True) if len(coeffs_hp) > 1 else []

# ----------------------------- Helpers ----------------------------- #
def poly_eval(coeffs, r):
    p = CHP(0)
    for c in coeffs:
        p = p * r + c
    return p

def relative_residual(coeffs, r):
    numerator = abs(poly_eval(coeffs, r))
    denom = sum(abs(c) * abs(r)**(len(coeffs) - i - 1) for i, c in enumerate(coeffs))
    return numerator / denom if denom != 0 else numerator

# ----------------------------- Triplet Computation (modular) ----------------------------- #
def compute_local_triplet(f_poly, r, m):
    """
    Compute the local triplet T = (a, m, δ, α) as defined in
    "A Triplet-Based Parameterization for the Local Asymptotic Characterization of Polynomial Roots"
    (Ralf Becker, 2026).

    δ is the characteristic deflection distance:
        δ = |α|^{-1/m}   where   α = P^{(m)}(r) / m!

    Returns: (r, m, delta, alpha)
    delta = None when α = 0 (theoretical degenerate case → δ = ∞)
    """
    if m < 1:
        raise ValueError("Multiplicity must be ≥ 1")

    # Safe m-th derivative (works for any degree/multiplicity)
    deriv_poly = f_poly
    for _ in range(m):
        deriv_poly = deriv_poly.derivative()

    alpha = deriv_poly(r) / factorial(m)

    abs_alpha = abs(alpha)
    if abs_alpha == 0:
        delta = None          # marker for δ = ∞ (degenerate case)
    else:
        # Keep full precision of RHP
        delta = abs_alpha ** (-RHP(1) / RHP(m))

    return r, m, delta, alpha

# ----------------------------- Print Roots + Local Triplets ----------------------------- #
print(f"\nPolynomial degree: {len(coeffs_hp)-1}")

if roots:
    # ----------------------------- Local Asymptotic Triplets (new) -----------------------------
    print("\nLocal Asymptotic Triplets T = (a, m, δ) — Characteristic Deflection Distance")
    print("Small δ ⇒ stiff/sharp root    |    Large δ ⇒ flat/extended contact region")
    print(f"{'#':<3} {'Root a':<32} {'m':<4} {'δ (deflection)':<22} {'Residual':<10}")
    print("-" * 110)
    print("" * 110)

    for i, (r, mult) in enumerate(roots, 1):
        _, _, delta, alpha = compute_local_triplet(fC, r, mult)
        root_str = r.n(digits=print_digits)
        alpha_str = str(alpha.n(digits=10))
        # delta = None means ∞ (no Infinity object needed)
        delta_str = "∞" if delta is None else str(delta.n(digits=14))
        res = relative_residual(coeffs_hp, r)

        print(f"{i:<3} {root_str:<32} {mult:<4} {delta_str:<22} {res:.2e}")
else:
    print("No roots (constant polynomial)")


# ----------------------------- Field Computation -----------------------------
def compute_root_field(roots_with_mult, N=600, use_global_scaling=False):
    centroids = np.array([complex(r) for r, _ in roots_with_mult])
    mults = np.array([float(m) for _, m in roots_with_mult])
    
    deltas = []
    for r, m in roots_with_mult:
        _, _, delta, _ = compute_local_triplet(fC, r, m)
        d = float(delta) if delta is not None else 1e9
        deltas.append(d)
    deltas = np.array(deltas)

    max_root = float(max(np.abs(centroids))) if len(centroids) > 0 else 1.0
    max_delta = float(max(deltas))

    if use_global_scaling and max_delta < 5e6:          # conservative threshold
        R = max([abs(c) + d for c, d in zip(centroids, deltas)] + [1.0]) * 1.08
    else:
        R = max_root * 1.85

    if R > 1e8 or not np.isfinite(R):
        R = max_root * 2.5 + 10
        print(f"⚠ Large δ detected → root-focused scaling (R={R:.2e})")

    print(f"Plot radius R = {R:.3e}  |  max|root| = {max_root:.3e}  |  maxδ = {max_delta:.3e}")

    # ... rest of the function stays the same as previous version ...
    xs = np.linspace(-R, R, N)
    ys = np.linspace(-R, R, N)
    X, Y = np.meshgrid(xs, ys)
    Z = X + 1j*Y

    min_dist = np.full(X.shape, np.inf, dtype=float)
    for a, d in zip(centroids, deltas):
        if d > 0:
            min_dist = np.minimum(min_dist, np.abs(Z - a) / d)
    
    dist_field = np.log10(np.clip(min_dist, 1e-28, None))

    # Newton flow (same as before)
    log_deriv = np.zeros(Z.shape, dtype=complex)
    EPS = 1e-25
    for a, m_val in zip(centroids, mults):
        dz = Z - a
        safe = np.where(np.abs(dz) < EPS, EPS, dz)
        log_deriv += m_val / safe

    with np.errstate(divide='ignore', invalid='ignore'):
        V = -1.0 / log_deriv
        mag = np.abs(V)
        mag_safe = np.where(mag > 0, mag, 1.0)
        flow_u = np.real(V) / mag_safe
        flow_v = np.imag(V) / mag_safe

    return xs, ys, dist_field, flow_u, flow_v, R


# ----------------------------- Plotting Function -----------------------------
def plot_root_field(roots_with_mult, fC, poly_str="", use_global_scaling=False, N=600):
    """
    Main plotting function - works well for both low and high degree polynomials.
    """
    xs, ys, dist, fu, fv, R = compute_root_field(
        roots_with_mult, N=N, use_global_scaling=use_global_scaling
    )
    
    fig, ax = plt.subplots(figsize=(12, 10.5))
    
    # Field
    im = ax.imshow(dist, extent=[xs[0], xs[-1], ys[0], ys[-1]],
                   origin='lower', cmap='viridis', aspect='equal')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label=r'$\log_{10} \left( \min_i |z - a_i| / \delta_i \right)$')
    
    # Newton flow
    ax.streamplot(xs, ys, fu, fv, density=1.35, color='black', 
                  linewidth=0.55, arrowsize=0.9)
    
    # Roots and δ-circles
    for r_sage, mult in roots_with_mult:
        r = complex(r_sage)
        _, _, delta, _ = compute_local_triplet(fC, r_sage, mult)
        delta_f = float(delta) if delta is not None else 0.0
        
        if 0 < delta_f < R * 1.8:
            circle = Circle((r.real, r.imag), delta_f,
                            fill=False, color='red', linestyle='--',
                            linewidth=1.6, alpha=0.7)
            ax.add_patch(circle)
        
        ax.scatter(r.real, r.imag, color='red', s=40 * mult**0.65, 
                   edgecolors='black', zorder=5, linewidth=0.8)
    
    # === Improved title handling ===
    if poly_str:
        eq = poly_str.strip()
        if not eq.endswith('= 0') and not eq.endswith('=0'):
            eq += " = 0"
        short_eq = eq[:130] + " ..." if len(eq) > 130 else eq
    else:
        short_eq = "Polynomial"
    
    ax.set_title("δ-Normalized Root Influence Field + Newton Flow\n" +
                 f"${short_eq}$", 
                 fontsize=11, pad=18)
    
    ax.set_xlabel("Re(z)")
    ax.set_ylabel("Im(z)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
# ----------------------------- Plotting ----------------------------- #
if roots:

     
    plot_root_field(roots, fC, poly_str=poly_str, 
                use_global_scaling=False, N=700)
    
    roots_np = np.array([complex(r) for r, _ in roots])
    multiplicities = [mult for _, mult in roots]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# ---- Complex Plane (with δ circles) ----
ax1.axhline(0, color='gray', lw=1)
ax1.axvline(0, color='gray', lw=1)

# Plot roots
for r, mult in zip(roots_np, multiplicities):
    ax1.scatter(r.real, r.imag, color='red', s=12*mult, zorder=5)

# Plot δ circles and compute extents
max_extent = 0.0
max_delta = 0.0
deltas_list = []

for r_np, (r_sage, mult) in zip(roots_np, roots):
    _, _, delta, _ = compute_local_triplet(fC, r_sage, mult)
    delta_float = float(delta) if delta is not None else 0.0
    deltas_list.append(delta_float)
    
    if delta_float > 0:
        circle = Circle((r_np.real, r_np.imag),
                        radius=delta_float,
                        fill=False,
                        linestyle='--',
                        linewidth=1.5,
                        edgecolor='blue',
                        alpha=0.75)
        ax1.add_patch(circle)
        
        extent = abs(r_np) + delta_float
        if extent > max_extent:
            max_extent = extent
        if delta_float > max_delta:
            max_delta = delta_float

ax1.set_title("Roots in Complex Plane\n(red dot size ∝ multiplicity | blue dashed = δ)")
ax1.set_xlabel("Re")
ax1.set_ylabel("Im")
ax1.grid(True)

# === AGGRESSIVE PADDING (recommended for now) ===
if len(roots_np) > 0:
    max_root = float(max(np.abs(roots_np)))
    
    # Generous padding to fully contain all δ circles
    plot_radius = (max_root + max_delta) * 1.28
    
    # Minimum reasonable plot size
    plot_radius = max(plot_radius, 2.5)
    
    ax1.set_xlim(-plot_radius, plot_radius)
    ax1.set_ylim(-plot_radius, plot_radius)

ax1.set_aspect('equal')

# Optional: nice auto-zoom with some padding
if len(roots_np) > 0:
    max_r = float(max(np.abs(roots_np))) * 1.15
    ax1.set_xlim(-max_r, max_r)
    ax1.set_ylim(-max_r, max_r)

    # ---- Real Line Plot (with fixes for high-degree / high-scaling) ----
    real_roots = [(r, mult) for (r, mult) in roots if abs(r.imag()) < RHP('1e-12')]
    if roots_np.size > 0:
        xmin = float(min(roots_np.real)) - 1.0
        xmax = float(max(roots_np.real)) + 1.0
    else:
        xmin, xmax = -5.0, 5.0
    x_vals_initial = np.linspace(xmin, xmax, 2000)
    x_list = x_vals_initial.tolist()
    for r, _ in real_roots:
        root_float = float(r.real())
        if not any(abs(root_float - x) < 1e-12 for x in x_list):
            x_list.append(root_float)
    x_list.sort()
    x_vals = np.array(x_list)
    y_vals = np.array([float(f(RHP(xx))) for xx in x_vals])

    max_abs = np.max(np.abs(y_vals)) if len(y_vals) > 0 else 1.0
    if np.isinf(max_abs) or max_abs > 1e300:
        print("NOTICE: Polynomial values exceed float range → clipping plot")
        y_vals = np.clip(y_vals, -1e300, 1e300)
        max_abs = 1e300

    ax2.plot(x_vals, y_vals, lw=1, label="f(x)")
    ax2.axhline(0, color='gray', lw=1)
    for r, mult in real_roots:
        ax2.scatter(float(r.real()), 0.0, color='red', s=10*mult, zorder=5)
    ax2.set_title("Polynomial on Real Line (size = multiplicity)")
    ax2.set_xlabel("x")
    ax2.set_ylabel("f(x)")
    ax2.grid(True)
    ax2.legend()
    init_ylim = ax2.get_ylim()

    # ----------------------------- Buttons -----------------------------
    linthresh = 1.0
    def set_linear(event):
        ax2.set_yscale('linear')
        ax2.set_ylim(init_ylim)
        fig.canvas.draw_idle()

    def set_symlog(event):
        ax2.set_yscale('symlog', linthresh=linthresh, linscale=1.0)
        print("→ Using symmetric symlog scale")
        fig.canvas.draw_idle()

    axlinear = plt.axes([0.8, 0.02, 0.1, 0.04])
    axsymlog = plt.axes([0.65, 0.02, 0.1, 0.04])
    b_linear = Button(axlinear, 'Linear')
    b_symlog = Button(axsymlog, 'Symlog')
    b_linear.on_clicked(set_linear)
    b_symlog.on_clicked(set_symlog)

    plt.tight_layout()
    plt.show()
else:
    print("No plot: polynomial is constant, no roots to display.")